# Bright and dark modes from SLURM MCWF

In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
if repo_root.name == "bright_dark_modes_mcwf":
    repo_root = repo_root.parents[2]
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

analysis_dir = repo_root / "slurm" / "analyses" / "bright_dark_modes_mcwf"
input_dir = analysis_dir / "output"

from common.plotting.utils import save_figure
from common.plotting import (
    plot_bloch_angles,
    plot_harmonic_sweep,
    plot_spin_components,
)
from parser.mcwf import TrajectoryEnsemble
from parser.mfe import MFESolverParameters
from parser.moments import MomentSeries, SimulationMetadata
from post_analysis import (
    compute_windowed_harmonic_analysis,
    compute_harmonic_quality,
    compute_j_modes_from_j_moments,
    compute_j_moments,
    compute_mean_generator_moments,
    extract_family_phase_series,
)
from slurm.trajectory_ensemble_io import load_trajectory_ensemble_artifact
from solvers.mfe import compute_mfe_j_moments, solve_mfe

## Input artifact

In [ ]:
# Use the array job ID printed by submit.sh.
run_name = "ARRAY_JOB_ID"
artifact_path = input_dir / run_name / "ensemble.pkl"
save_plots = False
output_dir = input_dir / "plots" / run_name

if not artifact_path.is_file():
    raise FileNotFoundError(f"Missing simulation artifact: {artifact_path}")
if save_plots:
    output_dir.mkdir(parents=True, exist_ok=True)

artifact_path

## MCWF moments and equivalent MFE solution

In [ ]:
artifact = load_trajectory_ensemble_artifact(artifact_path)
required_fields = {"ensemble", "simulation_metadata", "array_index", "ntraj"}
missing_fields = required_fields.difference(artifact)
if missing_fields:
    raise ValueError(
        f"{artifact_path} is missing fields: {sorted(missing_fields)}"
    )

ensemble = artifact["ensemble"]
metadata = artifact["simulation_metadata"]
if not isinstance(ensemble, TrajectoryEnsemble):
    raise TypeError(f"{artifact_path} does not contain a TrajectoryEnsemble.")
if not isinstance(metadata, SimulationMetadata):
    raise TypeError(f"{artifact_path} does not contain SimulationMetadata.")
if artifact["array_index"] != -1:
    raise ValueError(f"{artifact_path} is not a combined ensemble.")
if artifact["ntraj"] != ensemble.moments.num_trajectories:
    raise ValueError(f"{artifact_path} has inconsistent trajectory counts.")
if ensemble.metadata is None:
    raise ValueError(f"{artifact_path} has no ensemble metadata.")
if len(metadata.Ni) != 2:
    raise ValueError("This analysis requires two physical groups.")
if len(metadata.phase_protocol.family_phases) != 3:
    raise ValueError("This analysis requires the three-phase protocol.")

raw_moments = ensemble.moments
ntraj = raw_moments.num_trajectories
sectors = tuple(ensemble.metadata.sectors)
mcwf_moments = MomentSeries(t=raw_moments.t.copy(), metadata=metadata)
mcwf_moments.generator_moments = raw_moments
mcwf_moments.mean_generator_moments = compute_mean_generator_moments(
    raw_moments,
)
mcwf_moments.J = compute_j_moments(
    mcwf_moments.mean_generator_moments,
    metadata=mcwf_moments.metadata,
)
# Keep raw moments attached for whole-trajectory bootstrap resampling.
del artifact, ensemble, raw_moments

# MFE uses the saved MCWF physical inputs, protocol, and time grid.
mfe_moments = MomentSeries(t=mcwf_moments.t.copy(), metadata=metadata)
mfe_parameters = MFESolverParameters(
    Ni=metadata.Ni,
    omega_i=metadata.omega_groups,
    Gamma=metadata.Gamma,
    phase_protocol=metadata.phase_protocol,
)
mfe_result = solve_mfe(mfe_parameters, t_eval=mfe_moments.t)
if not mfe_result.success:
    raise RuntimeError(mfe_result.message)
mfe_moments.J = compute_mfe_j_moments(mfe_result)

mcwf_moments.J_modes = compute_j_modes_from_j_moments(
    mcwf_moments.J,
    omega_groups=mcwf_moments.metadata.omega_groups,
)

mfe_moments.J_modes = compute_j_modes_from_j_moments(
    mfe_moments.J,
    omega_groups=mfe_moments.metadata.omega_groups,
)

Gamma = metadata.Gamma
saved_time_step = 0.1 / Gamma
N_total = sum(metadata.Ni)
delta0 = metadata.phase_protocol.family_phases[1].delta
delta_factor = delta0 / (N_total * metadata.Gamma)
delta_label = rf"$\delta/(N\Gamma)={delta_factor:g}$"
print(f"Ni={metadata.Ni}, couplings={metadata.omega_groups}")
print(f"{ntraj} trajectories; {len(sectors)} initial sectors")

## Aggregated J angles and moments

In [ ]:
mcwf_J = mcwf_moments.J.aggregated
aggregated_angle_fig, aggregated_angle_axes = plot_bloch_angles(
    mcwf_moments.t,
    mcwf_J.theta,
    mcwf_J.phi,
    labels=('MCWF',),
    colour_family_index=0,
    shade_index=2,
    linestyle='-',
    phase_protocol=metadata.phase_protocol,
    title="Aggregated J-sphere angles",
)
aggregated_component_fig, aggregated_component_axes = plot_spin_components(
    mcwf_moments.t,
    mcwf_J.x,
    mcwf_J.y,
    mcwf_J.z,
    mcwf_J.length,
    x_sem=mcwf_J.x_sem,
    y_sem=mcwf_J.y_sem,
    z_sem=mcwf_J.z_sem,
    length_sem=mcwf_J.length_sem,
    labels=('MCWF',),
    colour_family_index=0,
    shade_index=2,
    linestyle='-',
    phase_protocol=metadata.phase_protocol,
    title="Aggregated J-sphere moments",
)

mfe_J = mfe_moments.J.aggregated
aggregated_angle_fig, aggregated_angle_axes = plot_bloch_angles(
    mfe_moments.t,
    mfe_J.theta,
    mfe_J.phi,
    labels=('MFE',),
    colour_family_index=0,
    shade_index=2,
    linestyle='--',
    axes=aggregated_angle_axes,
    phase_protocol=metadata.phase_protocol,
    title="Aggregated J-sphere angles",
)
aggregated_component_fig, aggregated_component_axes = plot_spin_components(
    mfe_moments.t,
    mfe_J.x,
    mfe_J.y,
    mfe_J.z,
    mfe_J.length,
    x_sem=mfe_J.x_sem,
    y_sem=mfe_J.y_sem,
    z_sem=mfe_J.z_sem,
    length_sem=mfe_J.length_sem,
    labels=('MFE',),
    colour_family_index=0,
    shade_index=2,
    linestyle='--',
    axes=aggregated_component_axes,
    phase_protocol=metadata.phase_protocol,
    title="Aggregated J-sphere moments",
)

# Keep time labels clear of the shared figure legends.
aggregated_angle_fig.supxlabel("")
aggregated_angle_axes[-1].set_xlabel(r"$\Gamma t$")
aggregated_component_fig.supxlabel("")
plt.setp(aggregated_component_axes[2:], xlabel=r"$\Gamma t$")

save_figure(aggregated_angle_fig, output_dir / "aggregated_j_angles.png" if save_plots else None)
save_figure(aggregated_component_fig, output_dir / "aggregated_j_moments.png" if save_plots else None)


## Group-resolved J angles and moments

In [ ]:
mcwf_J = mcwf_moments.J
mcwf_group_labels = ("MCWF group 1", "MCWF group 2")
group_angle_fig, group_angle_axes = plot_bloch_angles(
    mcwf_J.t,
    mcwf_J.theta,
    mcwf_J.phi,
    labels=mcwf_group_labels,
    shade_index=2,
    linestyle='-',
    phase_protocol=metadata.phase_protocol,
    title="Group-resolved J-sphere angles",
)
group_component_fig, group_component_axes = plot_spin_components(
    mcwf_J.t,
    mcwf_J.x,
    mcwf_J.y,
    mcwf_J.z,
    mcwf_J.length,
    x_sem=mcwf_J.x_sem,
    y_sem=mcwf_J.y_sem,
    z_sem=mcwf_J.z_sem,
    length_sem=mcwf_J.length_sem,
    labels=mcwf_group_labels,
    shade_index=2,
    linestyle='-',
    phase_protocol=metadata.phase_protocol,
    title="Group-resolved J-sphere moments",
)

mfe_J = mfe_moments.J
mfe_group_labels = ("MFE group 1", "MFE group 2")
group_angle_fig, group_angle_axes = plot_bloch_angles(
    mfe_J.t,
    mfe_J.theta,
    mfe_J.phi,
    labels=mfe_group_labels,
    shade_index=2,
    linestyle='--',
    axes=group_angle_axes,
    phase_protocol=metadata.phase_protocol,
    title="Group-resolved J-sphere angles",
)
group_component_fig, group_component_axes = plot_spin_components(
    mfe_J.t,
    mfe_J.x,
    mfe_J.y,
    mfe_J.z,
    mfe_J.length,
    x_sem=mfe_J.x_sem,
    y_sem=mfe_J.y_sem,
    z_sem=mfe_J.z_sem,
    length_sem=mfe_J.length_sem,
    labels=mfe_group_labels,
    shade_index=2,
    linestyle='--',
    axes=group_component_axes,
    phase_protocol=metadata.phase_protocol,
    title="Group-resolved J-sphere moments",
)

# Keep time labels clear of the shared figure legends.
group_angle_fig.supxlabel("")
group_angle_axes[-1].set_xlabel(r"$\Gamma t$")
group_component_fig.supxlabel("")
plt.setp(group_component_axes[2:], xlabel=r"$\Gamma t$")

save_figure(group_angle_fig, output_dir / "group_j_angles.png" if save_plots else None)
save_figure(group_component_fig, output_dir / "group_j_moments.png" if save_plots else None)


## Bright and dark modes

In [ ]:
mcwf_bright = mcwf_moments.J_modes.bright
mcwf_dark = mcwf_moments.J_modes.dark
mode_fig, mode_axes = plot_spin_components(
    mcwf_moments.J_modes.t,
    (mcwf_bright.x, mcwf_dark.x),
    (mcwf_bright.y, mcwf_dark.y),
    (mcwf_bright.z, mcwf_dark.z),
    (mcwf_bright.length, mcwf_dark.length),
    x_sem=(mcwf_bright.x_sem, mcwf_dark.x_sem),
    y_sem=(mcwf_bright.y_sem, mcwf_dark.y_sem),
    z_sem=(mcwf_bright.z_sem, mcwf_dark.z_sem),
    length_sem=(mcwf_bright.length_sem, mcwf_dark.length_sem),
    labels=("MCWF bright", "MCWF dark"),
    phase_protocol=mcwf_moments.metadata.phase_protocol,
    shade_index=2,
    linestyle="-",
    spin_symbol="s",
    title="MCWF and MFE bright/dark modes",
)

mfe_bright = mfe_moments.J_modes.bright
mfe_dark = mfe_moments.J_modes.dark
mode_fig, mode_axes = plot_spin_components(
    mfe_moments.J_modes.t,
    (mfe_bright.x, mfe_dark.x),
    (mfe_bright.y, mfe_dark.y),
    (mfe_bright.z, mfe_dark.z),
    (mfe_bright.length, mfe_dark.length),
    labels=("MFE bright", "MFE dark"),
    phase_protocol=mfe_moments.metadata.phase_protocol,
    shade_index=2,
    linestyle="--",
    axes=mode_axes,
    spin_symbol="s",
    title="MCWF and MFE bright/dark modes",
)

# Per-panel time labels leave the shared figure legend unobstructed.
mode_fig.supxlabel("")
plt.setp(mode_axes[2:], xlabel=r"$\Gamma t$")

save_figure(mode_fig, output_dir / "collective_modes.png" if save_plots else None)


## Phase-2 dark-mode traces

In [ ]:
mcwf_dark = mcwf_moments.J_modes.dark
phase_2_t, phase_2_series = extract_family_phase_series(
    mcwf_moments.J_modes.t,
    (mcwf_dark.x, mcwf_dark.y, mcwf_dark.z, mcwf_dark.length),
    mcwf_moments.metadata.phase_protocol,
    family_phase_index=1,
)
phase_2_dark_components = phase_2_series[:3]
phase_2_sems = (None, None, None, None)
if mcwf_dark.x_sem is not None:
    _, phase_2_sems = extract_family_phase_series(
        mcwf_moments.J_modes.t,
        (mcwf_dark.x_sem, mcwf_dark.y_sem, mcwf_dark.z_sem, mcwf_dark.length_sem),
        mcwf_moments.metadata.phase_protocol,
        family_phase_index=1,
    )
dark_fig, dark_axes = plot_spin_components(
    phase_2_t,
    *phase_2_series,
    x_sem=phase_2_sems[0],
    y_sem=phase_2_sems[1],
    z_sem=phase_2_sems[2],
    length_sem=phase_2_sems[3],
    labels=("MCWF dark",),
    phase_protocol=mcwf_moments.metadata.phase_protocol,
    colour_family_index=0,
    shade_index=1,
    spin_symbol="s",
    title="MCWF and MFE dark modes during phase 2",
)

mfe_dark = mfe_moments.J_modes.dark
mfe_phase_2_t, mfe_phase_2_series = extract_family_phase_series(
    mfe_moments.J_modes.t,
    (mfe_dark.x, mfe_dark.y, mfe_dark.z, mfe_dark.length),
    mfe_moments.metadata.phase_protocol,
    family_phase_index=1,
)
mfe_phase_2_dark_components = mfe_phase_2_series[:3]
dark_fig, dark_axes = plot_spin_components(
    mfe_phase_2_t,
    *mfe_phase_2_series,
    labels=("MFE dark",),
    phase_protocol=mfe_moments.metadata.phase_protocol,
    colour_family_index=1,
    shade_index=1,
    axes=dark_axes,
    spin_symbol="s",
    title="MCWF and MFE dark modes during phase 2",
)
plt.setp(dark_axes, xlim=(phase_2_t[0], phase_2_t[-1]))
dark_fig.supxlabel("")
plt.setp(dark_axes[2:], xlabel=r"$\Gamma t$")
save_figure(dark_fig, output_dir / "phase_2_dark_mode.png" if save_plots else None)


## Windowed dark-mode harmonic analysis


In [ ]:
window_duration = 200.0 / Gamma
step_duration = window_duration / 4
min_fit_r_squared = 0.9
minimum_cycles = 3.0
minimum_samples_per_period = 8.0

if np.max(np.diff(phase_2_t)) > saved_time_step * (1.0 + 1e-7):
    raise ValueError("The cached saved-time grid is too coarse for this analysis.")

dark_harmonic_analysis = compute_windowed_harmonic_analysis(
    phase_2_t,
    phase_2_dark_components,
    window_duration=window_duration,
    step_duration=step_duration,
    min_fit_r_squared=0.0,
)
harmonic_quality = compute_harmonic_quality(
    dark_harmonic_analysis,
    sample_spacing=phase_2_t[1] - phase_2_t[0],
    window_duration=window_duration,
    labels=("MCWF dark x", "MCWF dark y", "MCWF dark z"),
    min_fit_r_squared=min_fit_r_squared,
    minimum_cycles=minimum_cycles,
    minimum_samples_per_period=minimum_samples_per_period,
    verbose=True,
)

mfe_dark_harmonic_analysis = compute_windowed_harmonic_analysis(
    mfe_phase_2_t,
    mfe_phase_2_dark_components,
    window_duration=window_duration,
    step_duration=step_duration,
    min_fit_r_squared=0.0,
)
mfe_harmonic_quality = compute_harmonic_quality(
    mfe_dark_harmonic_analysis,
    sample_spacing=mfe_phase_2_t[1] - mfe_phase_2_t[0],
    window_duration=window_duration,
    labels=("MFE dark x", "MFE dark y", "MFE dark z"),
    min_fit_r_squared=min_fit_r_squared,
    minimum_cycles=minimum_cycles,
    minimum_samples_per_period=minimum_samples_per_period,
    verbose=True,
)


In [ ]:
harmonic_fig, harmonic_axes = plot_harmonic_sweep(
    Gamma * dark_harmonic_analysis.window_centers,
    dark_harmonic_analysis.fundamental_frequencies.T / Gamma,
    dark_harmonic_analysis.total_harmonic_distortions.T,
    dark_harmonic_analysis.rms_amplitudes.T,
    dark_harmonic_analysis.offsets.T,
    quality_masks=harmonic_quality.T,
    labels=("MCWF dark x", "MCWF dark y", "MCWF dark z"),
    parameter_label=r"Window-centre time $\Gamma t$",
    family_phase_index=1,
    shade_index=2,
    title="Windowed dark-mode harmonics",
)
harmonic_fig, harmonic_axes = plot_harmonic_sweep(
    Gamma * mfe_dark_harmonic_analysis.window_centers,
    mfe_dark_harmonic_analysis.fundamental_frequencies.T / Gamma,
    mfe_dark_harmonic_analysis.total_harmonic_distortions.T,
    mfe_dark_harmonic_analysis.rms_amplitudes.T,
    mfe_dark_harmonic_analysis.offsets.T,
    quality_masks=mfe_harmonic_quality.T,
    labels=("MFE dark x", "MFE dark y", "MFE dark z"),
    parameter_label=r"Window-centre time $\Gamma t$",
    family_phase_index=1,
    shade_index=2,
    linestyle="--",
    axes=harmonic_axes,
)
harmonic_axes[0].set_ylabel(r"$f_0/\Gamma$")
harmonic_axes[2].set_title("Background (fit offset / window mean)")
harmonic_fig.suptitle("Windowed dark-mode harmonics\n× marks fits that fail the display criteria", y=1.10, fontsize=14)
save_figure(harmonic_fig, output_dir / "dark_mode_harmonics.png" if save_plots else None)


## Early-time damped-oscillation fit


In [ ]:
from post_analysis.short_time_oscillations import compute_short_time_dark_mode_analysis

dark_decay_analysis = compute_short_time_dark_mode_analysis(
    mcwf_moments,
    duration=200.0 / Gamma,
    reference_frequencies=mfe_dark_harmonic_analysis.fundamental_frequencies,
    num_resamples=200,
    confidence_level=0.95,
    seed=1234,
    verbose=True,
)


In [ ]:
from common.plotting.harmonic_analysis import plot_damped_oscillations

mfe_dark = mfe_moments.J_modes.dark
decay_fig, decay_axes = plot_damped_oscillations(
    dark_decay_analysis.t,
    dark_decay_analysis.components,
    dark_decay_analysis.fits,
    labels=("Dark x", "Dark y", "Dark z"),
    sem=dark_decay_analysis.sem,
    reference_t=mfe_moments.t,
    reference_series=(mfe_dark.x, mfe_dark.y, mfe_dark.z),
    reference_label="MFE dark mode",
    time_scale=Gamma,
    time_label=r"Time since fit start $\Gamma(t-t_0)$",
)
save_figure(decay_fig, output_dir / "early_dark_mode_decay.png" if save_plots else None)


## Analysis summary

In [ ]:
{
    "artifact": str(artifact_path),
    "Ni": metadata.Ni,
    "ntraj": ntraj,
    "initial sectors": sectors,
    "delta/(N*Gamma)": delta_factor,
    "first moments (R, T, G, 8)": mcwf_moments.generator_moments.first_moments.shape,
    "packed second moments (R, T, K)": mcwf_moments.generator_moments.second_moments.shape,
    "mean first moments": mcwf_moments.mean_generator_moments.first_moments.shape,
    "group couplings": mcwf_moments.metadata.omega_groups,
    "MFE solve succeeded": mfe_result.success,
    "MCWF/MFE saved grids match": np.array_equal(mcwf_moments.t, mfe_moments.t),
    "phase-2 samples": phase_2_t.size,
    "harmonic window duration": window_duration,
    "harmonic step duration": step_duration,
    "minimum fit R-squared": min_fit_r_squared,
    "accepted harmonic windows": {
        "MCWF": harmonic_quality.sum(axis=0),
        "MFE": mfe_harmonic_quality.sum(axis=0),
    },
}
